# YouTube + Tubi QoE Report

Per-tier rendering of the concurrent YouTube + Tubi experiment (`run_direct_youtube_tubi_experiment.py`), one minute of synchronized playback per bandwidth tier with 100 ms latency and pfifo. Set `BANDWIDTH_MBPS` (3, 6, or 10) before running.

In [ ]:
from pathlib import Path
import json
import os
import math
import subprocess

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
import sys
REPO_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / '.git').exists())
sys.path.insert(0, str(REPO_ROOT / 'experiments/designated_experiments'))
from graph_output import install_graph_saver

BANDWIDTH_MBPS = int(os.environ.get('BANDWIDTH_MBPS', '3'))
result_candidates = [
    Path(f'youtube_tubi_experiment/{BANDWIDTH_MBPS}mbps'),
    REPO_ROOT / f'experiments/youtube_tubi_experiment/{BANDWIDTH_MBPS}mbps',
]
RESULT_DIR = next((p.resolve() for p in result_candidates if p.is_dir()), None)
if RESULT_DIR is None:
    raise FileNotFoundError(f'Could not locate the {BANDWIDTH_MBPS} Mbps result directory')

def load_qoe(app):
    path = RESULT_DIR / f'{app}_stats.jsonl'
    if not path.is_file():
        raise FileNotFoundError(f'Missing required QoE data: {path}')
    with path.open() as handle:
        return [json.loads(line) for line in handle if line.strip()]

youtube_qoe = load_qoe('youtube')
tubi_qoe = load_qoe('tubi')
start_time = min(youtube_qoe[0]['timestamp'], tubi_qoe[0]['timestamp'])
youtube_seconds = [row['timestamp'] - start_time for row in youtube_qoe]
tubi_seconds = [row['timestamp'] - start_time for row in tubi_qoe]
plt.style.use('seaborn-v0_8-whitegrid')
GRAPHS_DIR = install_graph_saver(plt, RESULT_DIR, [
    '01_download_throughput_by_application',
    '02_upload_throughput_by_application',
    '03_buffer_health_seconds',
    '04_cumulative_dropped_frame_rate',
    '05_video_resolution_over_time',
    '06_total_video_frames',
])
YOUTUBE_COLOR, TUBI_COLOR = '#ff0000', '#0f4bff'

ys = [row['stats'] for row in youtube_qoe]
ts = [row['stats'] for row in tubi_qoe]
summary = pd.DataFrame([{
    'Bandwidth': f'{BANDWIDTH_MBPS} Mbps',
    'YouTube final resolution': ys[-1].get('resolution'),
    'YouTube dropped frames': (ys[-1].get('dropped_video_frames', 0) or 0) - (ys[0].get('dropped_video_frames', 0) or 0),
    'YouTube total frames': ys[-1].get('total_video_frames'),
    'Tubi final resolution': ts[-1].get('resolution'),
    'Tubi dropped frames': (ts[-1].get('dropped_video_frames', 0) or 0) - (ts[0].get('dropped_video_frames', 0) or 0),
    'Tubi total frames': ts[-1].get('total_video_frames'),
}]).set_index('Bandwidth')
display(summary.round(3))


## Per-application throughput

Traffic is attributed from TLS/QUIC server names (SNI), with an IP-range fallback for YouTube's video CDN. YouTube: `youtube.com`, `googlevideo.com`, `ytimg.com`, `ggpht.com`, `gvt1.com`, plus Google's published edge/video-CDN IP ranges. Tubi: `tubitv.com`, `tubi.io`, `tubi.video`, `adrise.tv`. Any remaining bytes are shown separately as "Unclassified".

In [ ]:
PCAP = next(RESULT_DIR.glob('*.pcap'))
CLIENT_IP = '172.16.1.1'
window_start = min(youtube_qoe[0]['timestamp'], tubi_qoe[0]['timestamp'])
window_end = max(youtube_qoe[-1]['timestamp'], tubi_qoe[-1]['timestamp'])
window_seconds = window_end - window_start

sni_output = subprocess.run(
    ['tshark', '-r', str(PCAP), '-Y', 'tls.handshake.extensions_server_name',
     '-T', 'fields', '-e', 'ip.dst', '-e', 'tls.handshake.extensions_server_name'],
    check=True, capture_output=True, text=True,
).stdout
hosts_by_ip = {}
for line in sni_output.splitlines():
    fields = line.split('\t')
    if len(fields) >= 2 and fields[0] and fields[1]:
        hosts_by_ip.setdefault(fields[0], set()).update(h.lower() for h in fields[1].split(','))

youtube_markers = ('youtube.com', 'googlevideo.com', 'ytimg.com', 'ggpht.com', 'gvt1.com')
tubi_markers = ('tubitv.com', 'tubi.io', 'tubi.video', 'adrise.tv')
youtube_ips = {ip for ip, hosts in hosts_by_ip.items() if any(m in h for h in hosts for m in youtube_markers)}
tubi_ips = {ip for ip, hosts in hosts_by_ip.items() if any(m in h for h in hosts for m in tubi_markers)} - youtube_ips

import ipaddress
youtube_networks = [ipaddress.ip_network(cidr) for cidr in (
    '172.217.0.0/16', '142.250.0.0/15', '142.251.0.0/16', '74.125.0.0/16',
    '64.233.160.0/19', '173.194.0.0/16', '108.177.0.0/17', '216.58.0.0/16',
    '34.104.0.0/16',
)]

def classify_ip(ip):
    if ip in youtube_ips:
        return 'YouTube'
    if ip in tubi_ips:
        return 'Tubi'
    addr = ipaddress.ip_address(ip)
    if any(addr in net for net in youtube_networks):
        return 'YouTube'
    return 'Unclassified'

def classify_packets(direction_filter, remote_ip_field):
    output = subprocess.run(
        ['tshark', '-r', str(PCAP), '-Y', direction_filter, '-T', 'fields',
         '-e', 'frame.time_epoch', '-e', remote_ip_field, '-e', 'frame.len'],
        check=True, capture_output=True, text=True,
    ).stdout
    rows = []
    for line in output.splitlines():
        fields = line.split('\t')
        if len(fields) < 3 or not fields[0] or not fields[1] or not fields[2]:
            continue
        timestamp = float(fields[0])
        if not window_start <= timestamp <= window_end:
            continue
        remote_ip = fields[1].split(',')[0]
        frame_bytes = int(fields[2].split(',')[0])
        rows.append((timestamp, remote_ip, frame_bytes, classify_ip(remote_ip)))
    frame = pd.DataFrame(rows, columns=['timestamp', 'remote_ip', 'frame_bytes', 'application'])
    frame['second'] = (frame.timestamp - window_start).astype(int)
    return frame

download_packets = classify_packets(f'ip.dst == {CLIENT_IP}', 'ip.src')
upload_packets = classify_packets(f'ip.src == {CLIENT_IP}', 'ip.dst')

applications = ['YouTube', 'Tubi', 'Unclassified']
app_colors = {'YouTube': YOUTUBE_COLOR, 'Tubi': TUBI_COLOR, 'Unclassified': '#999999'}
bin_count = max(1, math.ceil(window_seconds))

def per_second_mbps(packets):
    app_bytes = (packets[packets.application.isin(applications)]
                 .groupby(['second', 'application']).frame_bytes.sum()
                 .unstack(fill_value=0)
                 .reindex(range(bin_count), fill_value=0)
                 .reindex(columns=applications, fill_value=0))
    return app_bytes * 8 / 1_000_000

download_mbps = per_second_mbps(download_packets)
upload_mbps = per_second_mbps(upload_packets)

def summarize(packets, app_mbps, label):
    rows = []
    for app in applications:
        total_bytes = packets.loc[packets.application == app, 'frame_bytes'].sum()
        rows.append({
            'application': app,
            f'{label} (MiB)': total_bytes / 2**20,
            'average during QoE window (Mbps)': total_bytes * 8 / window_seconds / 1_000_000,
            'peak 1-second bin (Mbps)': app_mbps[app].max(),
        })
    display(pd.DataFrame(rows).set_index('application').round(3))

print('Download:')
summarize(download_packets, download_mbps, 'downloaded')
print('Upload:')
summarize(upload_packets, upload_mbps, 'uploaded')


## Download throughput

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
for app in applications:
    ax.plot(download_mbps.index, download_mbps[app], marker='o', color=app_colors[app], label=app, alpha=(1.0 if app != 'Unclassified' else 0.6))
ax.axhline(BANDWIDTH_MBPS, color='#333333', linestyle='--', alpha=.7, label=f'Configured bottleneck ({BANDWIDTH_MBPS} Mbps)')
ax.set(title=f'Per-application Download Throughput — {BANDWIDTH_MBPS} Mbps', xlabel='Seconds from synchronized start', ylabel='Downloaded Mbit in each 1-second bin')
ax.legend()
plt.tight_layout()
plt.show()


## Upload throughput

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
for app in applications:
    ax.plot(upload_mbps.index, upload_mbps[app], marker='o', color=app_colors[app], label=app, alpha=(1.0 if app != 'Unclassified' else 0.6))
ax.set(title=f'Per-application Upload Throughput — {BANDWIDTH_MBPS} Mbps', xlabel='Seconds from synchronized start', ylabel='Uploaded Mbit in each 1-second bin')
ax.legend()
plt.tight_layout()
plt.show()


## Buffer health

In [ ]:
youtube_buffer = [row['stats'].get('buffer_ahead_secs', 0) or 0 for row in youtube_qoe]
tubi_buffer = [row['stats'].get('buffer_ahead_secs', 0) or 0 for row in tubi_qoe]
fig, ax = plt.subplots(figsize=(10, 4.8), dpi=120)
ax.plot(youtube_seconds, youtube_buffer, marker='o', color=YOUTUBE_COLOR, label='YouTube')
ax.plot(tubi_seconds, tubi_buffer, marker='o', color=TUBI_COLOR, label='Tubi')
ax.axhline(0, color='#ff7f7f', linestyle='--', alpha=.7)
ax.set(title=f'Buffer Health — {BANDWIDTH_MBPS} Mbps', xlabel='Seconds from synchronized start', ylabel='Buffer ahead of playhead (seconds)')
ax.legend(loc='best')
plt.tight_layout()
plt.show()


## Dropped frame rate

In [ ]:
def drop_percent(qoe):
    out = []
    for row in qoe:
        dropped = row['stats'].get('dropped_video_frames', 0) or 0
        total = row['stats'].get('total_video_frames', 0) or 0
        out.append(100 * dropped / max(1, total))
    return out

youtube_drop_pct = drop_percent(youtube_qoe)
tubi_drop_pct = drop_percent(tubi_qoe)
fig, ax = plt.subplots(figsize=(10, 4.8), dpi=120)
ax.plot(youtube_seconds, youtube_drop_pct, marker='o', color=YOUTUBE_COLOR, label='YouTube')
ax.plot(tubi_seconds, tubi_drop_pct, marker='o', color=TUBI_COLOR, label='Tubi')
ax.set(title=f'Cumulative Dropped Frame Rate — {BANDWIDTH_MBPS} Mbps', xlabel='Seconds from synchronized start', ylabel='Dropped frames (% of decoded frames so far)')
ax.legend(loc='best')
plt.tight_layout()
plt.show()


## Video resolution over time

In [ ]:
youtube_height = [row['stats'].get('video_height', 0) or 0 for row in youtube_qoe]
youtube_width = [row['stats'].get('video_width', 0) or 0 for row in youtube_qoe]
tubi_height = [row['stats'].get('video_height', 0) or 0 for row in tubi_qoe]
tubi_width = [row['stats'].get('video_width', 0) or 0 for row in tubi_qoe]

fig, ax = plt.subplots(figsize=(10, 4.8), dpi=120)
ax.step(youtube_seconds, youtube_height, where='post', color=YOUTUBE_COLOR, linewidth=2, label='YouTube')
ax.scatter(youtube_seconds, youtube_height, color=YOUTUBE_COLOR, s=24)
ax.step(tubi_seconds, tubi_height, where='post', color=TUBI_COLOR, linewidth=2, label='Tubi')
ax.scatter(tubi_seconds, tubi_height, color=TUBI_COLOR, s=24)

resolution_labels = {}
for width, height in zip(youtube_width + tubi_width, youtube_height + tubi_height):
    if width and height:
        resolution_labels.setdefault(height, set()).add(f'{width}x{height}')
tick_heights = sorted(resolution_labels)
if tick_heights:
    ax.set_yticks(tick_heights, [' / '.join(sorted(resolution_labels[h])) for h in tick_heights])
ax.set(title=f'Video Resolution Over Time — {BANDWIDTH_MBPS} Mbps', xlabel='Seconds from synchronized start', ylabel='Rendered resolution')
ax.legend(loc='best')
plt.tight_layout()
plt.show()


## Total video frames

In [ ]:
youtube_frames = [row['stats'].get('total_video_frames', 0) or 0 for row in youtube_qoe]
tubi_frames = [row['stats'].get('total_video_frames', 0) or 0 for row in tubi_qoe]
fig, ax = plt.subplots(figsize=(10, 4.8), dpi=120)
ax.plot(youtube_seconds, youtube_frames, marker='o', color=YOUTUBE_COLOR, label='YouTube')
ax.plot(tubi_seconds, tubi_frames, marker='o', color=TUBI_COLOR, label='Tubi')
ax.set(title=f'Total Video Frames — {BANDWIDTH_MBPS} Mbps', xlabel='Seconds from synchronized start', ylabel='Cumulative total video frames')
ax.legend(loc='best')
plt.tight_layout()
plt.show()
